#  검색 성능 향상을 위한 기법 

- **학습 목표:** 키워드 검색과 하이브리드 검색 방식을 실습하고 비교 분석한다

---

## 환경 설정 및 준비

### 사전 준비

**필수 데이터 파일:**
- `data/*_KR.md` - 한국어 문서 데이터 (테슬라, 리비안 등)
- `data/testset.xlsx` - 검색 성능 평가용 테스트셋

**필수 패키지:**
- `langchain-community` - BM25Retriever, EnsembleRetriever 등
- `langchain-chroma` - Chroma 벡터 저장소
- `kiwipiepy` - 한국어 형태소 분석기
- `rank_bm25` - BM25 알고리즘 구현
- `ranx-k` - 한국어 최적화 검색 성능 평가 라이브러리 (Kiwi 형태소 분석기 기반)

**패키지 설치:**
```bash
uv pip install ranx-k kiwipiepy rank_bm25
```

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

`(3) langfuse handler 설정`

In [ ]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

---

## RAG 검색기

1. **의미론적 검색 (Semantic Search)**
    - Vector Store를 기반으로 한 검색 방식으로, 텍스트의 의미적 유사성을 고려하여 검색을 수행함
    - 임베딩 벡터 간의 유사도를 계산하여 의미적으로 관련성이 높은 문서를 찾아내는 특징이 있음
    - 동의어나 문맥적 의미를 파악할 수 있어 자연어 질의에 효과적임

1. **키워드 검색 (Keyword Search)**
    - BM25와 같은 전통적인 검색 알고리즘을 사용하여 키워드 매칭을 기반으로 검색을 수행함
    - 정확한 단어나 구문 매칭에 강점이 있으며, 계산 효율성이 높은 특징을 가짐
    - 직접적인 키워드 일치를 찾는 데 유용하나, 의미적 유사성을 파악하는 데는 한계가 있음

1. **하이브리드 검색 (Hybrid Search)**
    - 키워드 기반 검색과 의미론적 검색을 결합한 방식으로, EnsembleRetriever를 통해 구현됨
    - 두 검색 방식의 장점을 활용하여 더 정확하고 포괄적인 검색 결과를 제공함
    - 정확한 키워드 매칭과 의미적 연관성을 모두 고려하여 검색 성능을 향상시키는 특징이 있음

### 1) **Semantic Search** (의미론적 검색) 

- **의미론적 검색**은 텍스트의 **벡터 표현**을 활용해 의미적 유사성 기반 검색 수행
- **Vector Store**에 저장된 임베딩 벡터 간 **유사도 계산**으로 관련 문서 검색
- 검색어와 문서 간의 **문맥적 의미**와 **동의어 관계**를 효과적으로 파악
- **자연어 질의**에 강점을 보이며 기존 키워드 검색의 한계를 보완
- 전통적인 검색 방식과 달리 **의미 기반 매칭**으로 더 정확하고 포괄적인 검색 결과 제공

`(1) 벡터 저장소 초기화`
- cosine distance 기준으로 인덱싱 

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 데이터 로드
def load_text_files(txt_files):
    data = []

    for text_file in txt_files:
        loader = TextLoader(text_file, encoding='utf-8')
        data += loader.load()

    return data

korean_txt_files = glob(os.path.join('data', '*_KR.md')) 
korean_data = load_text_files(korean_txt_files)

# 문장을 구분하여 분할 - 정규표현식 사용 (문장 구분자: 마침표, 느낌표, 물음표 다음에 공백이 오는 경우)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",    # TikToken 인코더 이름
    separators=['\n\n', '\n', r'(?<=[.!?])\s+'],   # 구분자
    chunk_size=300,            # 문서 분할 크기
    chunk_overlap=50,          # 문서 분할 중첩  
    is_separator_regex=True,      # 구분자가 정규식인지 여부
    keep_separator=True,          # 구분자 유지 여부
)

korean_chunks = text_splitter.split_documents(korean_data)

print("한국어 청크 수:", len(korean_chunks))

In [ ]:
for i, doc in enumerate(korean_chunks):
    print(f"[{i}]", doc.page_content)
    print("="*200)

In [ ]:
doc.metadata

In [ ]:
from langchain_core.documents import Document
# Document 객체에 메타데이터 추가

korean_docs = []

for chunk in korean_chunks:
    doc = Document(page_content=chunk.page_content, metadata=chunk.metadata)
    doc.metadata['company'] = '테슬라(Tesla)' if 'Tesla' in doc.metadata['source'] else '리비안(Rivian)'
    doc.metadata['language'] = 'ko'
    doc.page_content = f"[출처] 이 문서는 {doc.metadata['company']}에 대한 문서입니다.\n----------------------------------\n{doc.page_content}"
    korean_docs.append(doc)

print("한국어 문서 수:", len(korean_docs))
print(korean_docs[0].metadata)
print(korean_docs[-1].metadata)
print("="*200)
print(korean_docs[0].page_content)

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# OpenAI Embeddings 모델을 로드
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Chroma 벡터 저장소 생성하기
chroma_db = Chroma.from_documents(
    documents=korean_docs,
    embedding=embeddings,    
    collection_name="db_korean_cosine_metadata", 
    persist_directory="./chroma_db",
    collection_metadata = {'hnsw:space': 'cosine'}, # l2, ip, cosine 중에서 선택 
)

`(2) 벡터 저장소 로드`
- 미리 인덱싱해서 저장해 둔 저장소를 가져와서 사용
- 이때 기존에 사용한 임베딩 모델을 초기화 필요

In [ ]:
# 벡터 저장소 로드 
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

`(3) Semantic Search 실행`
- 벡터 저장소 검색기 객체 활용
- 임베딩 벡터 간의 유사도를 기반으로 문서 검색

In [ ]:
# 검색기 지정하여 테스트 
chroma_k_retriever = chroma_db.as_retriever(
    search_kwargs={"k": 2},
)

query = "리비안은 언제 사업을 시작했나요?"
retrieved_docs = chroma_k_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

### 2) **Keyword Search** (키워드 검색) 

- **키워드 검색**은 **BM25** 등 전통적 알고리즘 기반의 단어 매칭 방식
- 정확한 **단어/구문 매칭**에 강점이 있으며 **계산 효율성**이 우수함
- **직접적인 키워드** 검색에는 효과적이나 의미적 연관성 파악에는 제한적
- 구현이 단순하고 **처리 속도가 빠르다**는 장점이 있음
- 정확한 키워드 매칭이 필요한 경우에 적합하나 **의미론적 검색의 보완**이 필요함

`(1) BM25 검색기 생성`

- BM25: TF-IDF (Term Frequency-Inverse Document Frequency)의 확장된 버전
- `rank_bm25` 설치

In [ ]:
# 벡터 저장소에 저정한 문서 객체를 로드하여 확인
chroma_db.get().keys()

In [ ]:
# BM25 검색기 생성을 위해 문서 객체를 로드
documents = chroma_db.get()["documents"]
metadatas = chroma_db.get()["metadatas"]

# Document 객체로 변환
from langchain_core.documents import Document
docs = [Document(page_content=content, metadata=meta) for content, meta in zip(documents, metadatas)]

print("문서의 수:" , len(docs))
print("=" * 200)
for doc in docs[:3]:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("-" * 200)

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 검색기 생성
bm25_retriever = BM25Retriever.from_documents(docs)

# BM25 검색기를 사용하여 검색
query = "리비안은 언제 사업을 시작했나요?"

retrieved_docs = bm25_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

In [ ]:
# BM25 점수를 확인 - "리비안은" 라는 단어가 쿼리에 포함되어 있지 않아 검색 결과가 없음
query = "리비안은 언제 사업을 시작했나요?"
tokenized_query = query.split()
print(tokenized_query)
print("="*200)

# 문서의 BM25 점수 확인
doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)

# 문서의 BM25 점수를 내림차순으로 정렬
doc_scores_sorted = sorted(enumerate(doc_scores), key=lambda x: x[1], reverse=True)

# 상위 5개 문서의 인덱스와 점수를 출력
for idx, score in doc_scores_sorted[:5]:
    print(f"[{idx}] {docs[idx].page_content}")
    print(f"BM25 Score: {score}")
    print("-"*200)

In [ ]:
# 같은 의미를 갖는 쿼리로 변경하여 다시 검색 
query = "리비안이 설립된 연도는?"

retrieved_docs = bm25_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

In [ ]:
# BM25 점수를 확인 - "리비안이" 라는 단어가 쿼리에 포함되어 있어 검색 결과가 있음
query = "리비안이 설립된 연도는?"
tokenized_query = query.split()
print(tokenized_query)
print("="*200)

# 문서의 BM25 점수 확인
doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)

# 문서의 BM25 점수를 내림차순으로 정렬
doc_scores_sorted = sorted(enumerate(doc_scores), key=lambda x: x[1], reverse=True)

# 상위 5개 문서의 인덱스와 점수를 출력
for idx, score in doc_scores_sorted[:5]:
    print(f"[{idx}] {docs[idx].page_content}")
    print(f"BM25 Score: {score}")
    print("-"*200)

`(2) kiwi 한국어 토크나이저`
- `kiwipiepy` 설치 필요 (pip install kiwipiepy / uv add kiwipiepy)

In [ ]:
# 한국어 토크나이저를 사용하여 문장을 토큰화
from kiwipiepy import Kiwi

kiwi = Kiwi()

print(kiwi.analyze("리비안은 언제 사업을 시작했나요?"))
print("="*200)
print(kiwi.analyze("리비안이 설립된 연도는?"))

In [ ]:
print(kiwi.tokenize("테슬라는 언제 설립되었나요?"))
print("="*200)
print(kiwi.tokenize("테슬라가 설립된 연도는?"))

In [ ]:
# 단어를 추가 
kiwi.add_user_word('리비안', 'NNP')  # NNP: 고유명사

print(kiwi.analyze("리비안은 언제 사업을 시작했나요?"))
print("="*200)
print(kiwi.analyze("리비안이 설립된 연도는?"))

In [ ]:
# 한국어 토크나이저를 사용하여 문장을 토큰화하는 함수를 정의 

def bm25_process_func(text, kiwi_model=Kiwi()):
    """
    BM25Retriever에서 사용할 전처리 함수
    한국어 토크나이저를 사용하여 문장을 토큰화
    :param text: 토큰화할 문장
    :param kwii_model: Kiwi 객체 (from kiwipiepy import Kiwi 사용)
    """

    return [t.form for t in kiwi_model.tokenize(text)]


# BM25Retriever 객체 생성
bm25_retriever = BM25Retriever.from_documents(
    documents=docs,
    preprocess_func=lambda x: bm25_process_func(x, kiwi_model=kiwi),
    )

# 이전에 사용한 검색어를 입력하여 문서를 검색
query = "리비안이 설립된 연도는?"

retrieved_docs = bm25_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

In [ ]:
# BM25 점수를 확인
query = "리비안이 설립된 연도는?"

tokenized_query = [t.form for t in kiwi.tokenize(query)]
print(tokenized_query)
print("="*200)

# 문서의 BM25 점수 확인
doc_scores = bm25_retriever.vectorizer.get_scores(tokenized_query)

# 문서의 BM25 점수를 내림차순으로 정렬
doc_scores_sorted = sorted(enumerate(doc_scores), key=lambda x: x[1], reverse=True)

# 상위 5개 문서의 인덱스와 점수를 출력
for idx, score in doc_scores_sorted[:5]:
    print(f"[{idx}] {docs[idx].page_content}")
    print(f"BM25 Score: {score}")
    print("-"*200)

### 3) **Hybrid Search** (하이브리드 검색) 

- **하이브리드 검색**은 **키워드 검색**과 **의미론적 검색**을 **EnsembleRetriever**로 통합
- 정확한 **키워드 매칭**과 **의미적 유사성**을 동시에 고려하여 검색 수행
- 두 검색 방식의 **장점을 결합**하여 더 포괄적이고 정확한 결과 도출
- 검색 성능 향상을 위해 각 방식의 **가중치 조정**이 가능함
- 키워드와 의미 기반 검색의 **시너지 효과**로 더 향상된 검색 성능 실현 가능

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

# 앙상블 검색기 생성
ensemble_retrievers = [chroma_k_retriever, bm25_retriever]

ensemble_retriever = EnsembleRetriever(
    retrievers=ensemble_retrievers, 
    weights=[0.5, 0.5]          # 각 검색기의 가중치
)

# 검색기를 사용하여 검색
query = "리비안이 설립된 연도는?"

retrieved_docs = ensemble_retriever.invoke(query)

for doc in retrieved_docs:
    print(f"{doc.page_content} [출처: {doc.metadata['source']}]")
    print("="*200)

### Reciprocal Rank Fusion (RRF)

`EnsembleRetriever`는 내부적으로 **Reciprocal Rank Fusion (RRF)** 알고리즘을 사용하여 여러 검색기의 결과를 통합합니다.

**RRF 알고리즘 원리:**
- 각 검색기에서 문서의 순위(rank)를 기반으로 점수 계산
- RRF 점수 공식: `score = Σ (1 / (k + rank))` (k는 상수, 보통 60)
- 여러 검색기에서 높은 순위를 받은 문서가 최종적으로 상위에 배치됨

In [ ]:
from collections import defaultdict
from langchain_core.documents import Document

def reciprocal_rank_fusion(
    results_list: list[list[Document]], 
    weights: list[float] = None,
    k: int = 60
) -> list[Document]:
    """
    Reciprocal Rank Fusion (RRF) 알고리즘 직접 구현
    
    Args:
        results_list: 여러 검색기 결과 리스트 [[검색기1 결과], [검색기2 결과], ...]
        weights: 각 검색기의 가중치 (기본값: 동일 가중치)
        k: RRF 상수 (기본값 60, 논문 권장값)
    
    Returns:
        재순위화된 문서 리스트
    """
    # 기본 가중치 설정 (동일 가중치)
    if weights is None:
        weights = [1.0] * len(results_list)
    
    # 각 문서의 RRF 점수 계산
    rrf_scores = defaultdict(float)
    doc_map = {}  # 문서 내용 저장
    
    for weight, results in zip(weights, results_list):
        for rank, doc in enumerate(results, start=1):
            # 문서 식별을 위해 page_content 사용 (실제로는 고유 ID 권장)
            doc_id = doc.page_content
            # 가중치가 적용된 RRF 점수 누적
            rrf_scores[doc_id] += weight * (1 / (k + rank))
            doc_map[doc_id] = doc
    
    # 점수 기준 내림차순 정렬
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    return [doc_map[doc_id] for doc_id, score in sorted_docs]


# RRF 직접 구현 사용 예시
query = "리비안이 설립된 연도는?"

# 각 검색기에서 결과 가져오기
bm25_results = bm25_retriever.invoke(query)
vector_results = chroma_k_retriever.invoke(query)

# RRF로 결과 통합 (50:50 가중치)
fused_results = reciprocal_rank_fusion(
    results_list=[vector_results, bm25_results],
    weights=[0.5, 0.5]
)

print("=" * 80)
print("RRF 직접 구현 결과:")
print("=" * 80)
for i, doc in enumerate(fused_results[:3], 1):
    print(f"\n[{i}] {doc.page_content[:150]}...")
    print(f"    출처: {doc.metadata.get('source', 'N/A')}")

---

## 검색 성능 평가

### 1) **테스트 데이터** 

- 합성된 데이터는 **품질 검증**과 **수동 수정** 과정을 거쳐 정제
- 테스트용 데이터는 **다양한 유형의 질문**과 **답변 패턴**을 포함해야 함
- 신뢰할 수 있는 검색 성능 평가를 위해 **고품질 테스트 데이터** 확보가 중요함

In [ ]:
# 기존에 생성해 둔 테스트셋 로드
# 테스트셋 로드
import pandas as pd
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

### 2) **Information Retrieval 평가지표**

`ranx-k`는 한국어에 최적화된 검색 성능 평가 라이브러리입니다.

**특징:**
- **Kiwi 형태소 분석기** 기반 텍스트 유사도 매칭
- 문서 ID가 아닌 **텍스트 내용** 기반 평가 (RAG 시스템에 적합)
- Hit Rate, MRR, MAP, NDCG 등 다양한 평가 지표 지원

**패키지 설치:**
```bash
pip install ranx-k
# 또는
uv add ranx-k
```

`(1) 키워드 검색 (Kiwi 토크나이저 + BM25)`

- 한국어에 최적화된 BM25 검색기를 생성합니다
- Kiwi 형태소 분석기를 전처리 함수로 사용합니다

In [ ]:
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# Kiwi 한국어 형태소 분석기 초기화
kiwi = Kiwi()

# 사용자 정의 단어 추가
kiwi.add_user_word('리비안', 'NNP')  # 고유명사
kiwi.add_user_word('테슬라', 'NNP')  # 고유명사

# 한국어 토크나이저 전처리 함수
def kiwi_preprocess_func(text):
    """Kiwi 형태소 분석기를 사용한 토큰화"""
    return [t.form for t in kiwi.tokenize(text)]

# BM25 검색기 생성 (Kiwi 토크나이저 적용)
retriever_bm25_kiwi = BM25Retriever.from_documents(
    documents=korean_docs,
    preprocess_func=kiwi_preprocess_func,
    k=3,
)

print(f"✅ BM25 검색기 생성 완료: {len(korean_docs)}개 문서 인덱싱")

In [ ]:
# BM25 검색기를 사용하여 문서 검색
question = df_qa_test['user_input'].iloc[0]
print("질문:", question)
print("="*200)
context = df_qa_test['reference_contexts'].iloc[0]
print("관련 문서:", context)
print("="*200)

# BM25 검색
retrieved_docs = retriever_bm25_kiwi.invoke(question)

# 검색 결과 출력 
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['company']}]")
    print("-"*200)

In [ ]:
from ranx_k.evaluation import evaluate_with_ranx_similarity

# 평가 데이터 준비: 질문 리스트와 정답 문서 리스트
questions = df_qa_test['user_input'].tolist()

# 정답 문서를 Document 객체 리스트로 변환
reference_contexts = []
for contexts in df_qa_test['reference_contexts']:
    docs = [Document(page_content=ctx) for ctx in eval(contexts)]
    reference_contexts.append(docs)

print(f"평가 데이터: {len(questions)}개 질문")

In [ ]:
# BM25 검색기 평가 (ranx-k 사용)
retriever_bm25_kiwi.k = 3

results_bm25 = evaluate_with_ranx_similarity(
    retriever=retriever_bm25_kiwi,
    questions=questions,
    reference_contexts=reference_contexts,
    k=3,
    method='kiwi_rouge',  # Kiwi 형태소 분석기 기반 ROUGE 점수
    similarity_threshold=0.8,
)

print("BM25 검색기 평가 결과 (ranx-k):")
print(f"  Hit Rate@3: {results_bm25.get('hit_rate@3', 0):.3f}")
print(f"  MRR: {results_bm25.get('mrr', 0):.3f}")
print(f"  MAP@3: {results_bm25.get('map@3', 0):.3f}")
print(f"  NDCG@3: {results_bm25.get('ndcg@3', 0):.3f}")

`(2) 시맨틱 검색 (Chroma 벡터저장소)`

In [ ]:
# Chroma 검색기 평가 (ranx-k 사용)
retriever_chroma = chroma_db.as_retriever(search_kwargs={"k": 3})

results_chroma = evaluate_with_ranx_similarity(
    retriever=retriever_chroma,
    questions=questions,
    reference_contexts=reference_contexts,
    k=3,
    method='kiwi_rouge',
    similarity_threshold=0.8,
)

print("Chroma 검색기 평가 결과 (ranx-k):")
print(f"  Hit Rate@3: {results_chroma.get('hit_rate@3', 0):.3f}")
print(f"  MRR: {results_chroma.get('mrr', 0):.3f}")
print(f"  MAP@3: {results_chroma.get('map@3', 0):.3f}")
print(f"  NDCG@3: {results_chroma.get('ndcg@3', 0):.3f}")

---

## 연습문제

다음 연습문제를 통해 키워드 검색과 하이브리드 검색에 대한 이해를 확인해 보세요.

### 문제 1: BM25 검색기 생성

아래 코드의 빈칸을 채워 BM25 검색기를 생성하고 상위 3개 문서를 검색하세요.

In [ ]:
from langchain_community.retrievers import ____  # 힌트: BM25Retriever

# BM25 검색기 생성
bm25_retriever = ____(docs)  # 힌트: BM25Retriever.from_documents 메서드 사용

# 상위 k개 설정
bm25_retriever.k = ____  # 힌트: 3

# 검색 수행
query = "테슬라 전기차 배터리"
results = bm25_retriever.____(query)  # 힌트: invoke 메서드

print(f"검색 결과: {len(results)}개")
for doc in results:
    print(f"- {doc.page_content[:50]}...")

### 문제 2: 하이브리드 검색 (EnsembleRetriever)

아래 코드의 빈칸을 채워 BM25와 벡터 검색을 결합한 하이브리드 검색기를 생성하세요.

In [ ]:
from langchain_classic.retrievers import ____  # 힌트: EnsembleRetriever

# 하이브리드 검색기 생성
# BM25와 벡터 검색기를 결합, 각각 50%씩 가중치 부여
ensemble_retriever = ____(
    retrievers=[bm25_retriever, chroma_retriever],  # 두 검색기 리스트
    weights=[____, ____]  # 힌트: 각각 0.5
)

# 검색 수행
query = "테슬라 전기차 배터리"
results = ensemble_retriever.____(query)  # 힌트: invoke 메서드

print(f"하이브리드 검색 결과: {len(results)}개")

### 문제 3: 하이브리드 검색 성능 평가

**목표:**
- `EnsembleRetriever`를 사용하여 하이브리드 검색기를 생성합니다
- `ranx-k`를 사용하여 테스트 데이터셋에 대한 검색 성능을 평가합니다

**힌트:**
- `EnsembleRetriever`는 여러 검색기를 결합하여 사용합니다
- `retrievers` 매개변수에 검색기 리스트를 전달합니다 (예: `[retriever1, retriever2]`)
- `weights` 매개변수로 각 검색기의 가중치를 조정합니다 (예: `[0.5, 0.5]`)
- `evaluate_with_ranx_similarity` 함수로 평가를 수행합니다

In [ ]:
# 하이브리드 검색기 설정
# TODO: retriever_bm25_kiwi와 retriever_chroma_db를 결합하여 EnsembleRetriever 생성
# - retrievers 매개변수에 두 검색기를 리스트로 전달
# - weights 매개변수로 각 검색기의 가중치 설정 (예: [0.5, 0.5])

ensemble_retriever = None

In [ ]:
# 하이브리드 검색기 평가 (ranx-k 사용)
# TODO: evaluate_with_ranx_similarity 함수로 앙상블 검색기 평가
# 힌트:
# results_ensemble = evaluate_with_ranx_similarity(
#     retriever=ensemble_retriever,
#     questions=questions,
#     ground_truths=ground_truths,
#     k=3,
#     similarity_threshold=0.7
# )

results_ensemble = None